# Практикум 10. Синтаксический парсинг

Сопровождает интерактивную лабораторию №10 («Деревья зависимостей и составляющих»). Ноутбук самодостаточен: первая ячейка создаёт файл `nlp_common.py` с учебным корпусом русского языка (шаблонные предложения с морфологической разметкой) и вспомогательными функциями. Запускайте ячейки сверху вниз. Для реальной задачи замените учебный корпус своим.

In [ ]:
%%writefile nlp_common.py
"""Учебный модуль: генераторы корпусов для русского языка."""
import random, re
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
plt.rcParams["figure.dpi"] = 110; plt.rcParams["axes.grid"] = True; plt.rcParams["grid.alpha"] = .25
LANG = "ru"
NOUN = {  # лемма: (им., вин., предл.)
 "мама": ("мама","маму","маме"), "девочка": ("девочка","девочку","девочке"), "студентка": ("студентка","студентку","студентке"),
 "учительница": ("учительница","учительницу","учительнице"), "книга": ("книга","книгу","книге"), "газета": ("газета","газету","газете"),
 "песня": ("песня","песню","песне"), "картина": ("картина","картину","картине"), "школа": ("школа","школу","школе"),
 "библиотека": ("библиотека","библиотеку","библиотеке"), "комната": ("комната","комнату","комнате"), "улица": ("улица","улицу","улице")}
SUBJ = ["мама","девочка","студентка","учительница"]; OBJ = ["книга","газета","песня","картина"]; PLACE = ["школа","библиотека","комната","улица"]
ADJ_NOM = {"умный":"умная","добрый":"добрая","молодой":"молодая","весёлый":"весёлая"}
ADJ_ACC = {"новый":"новую","интересный":"интересную","красивый":"красивую","старый":"старую"}
VERB = {"читать":"читает","писать":"пишет","любить":"любит","видеть":"видит"}

def make_sentences(n=500, seed=0):
    """Предложения с морфологической разметкой: tokens, tags (UPOS), lemmas."""
    r = random.Random(seed); out = []
    for _ in range(n):
        t, g, l = [], [], []
        if r.random() < .5:
            a = r.choice(list(ADJ_NOM)); t.append(ADJ_NOM[a]); g.append("ADJ"); l.append(a)
        s = r.choice(SUBJ); t.append(NOUN[s][0]); g.append("NOUN"); l.append(s)
        v = r.choice(list(VERB)); t.append(VERB[v]); g.append("VERB"); l.append(v)
        if r.random() < .5:
            a = r.choice(list(ADJ_ACC)); t.append(ADJ_ACC[a]); g.append("ADJ"); l.append(a)
        o = r.choice(OBJ); t.append(NOUN[o][1]); g.append("NOUN"); l.append(o)
        if r.random() < .4:
            p = r.choice(PLACE); t += ["в", NOUN[p][2]]; g += ["ADP", "NOUN"]; l += ["в", p]
        out.append(dict(tokens=t, tags=g, lemmas=l))
    return out

def all_forms():
    """Все формы существительных с эталонной леммой: [(форма, лемма)]."""
    return sorted({(f, k) for k, v in NOUN.items() for f in v})

_A = ["кошка","собака","лошадь","корова"]; _F = ["рыбу","мясо","хлеб","сыр"]; _D = ["молоко","воду","чай","сок"]; _V = ["машина","автобус","поезд","трамвай"]
def make_topic_sentences(n=2000, seed=0):
    r = random.Random(seed); fr = [lambda: [r.choice(_A),"ест",r.choice(_F)], lambda: [r.choice(_A),"пьёт",r.choice(_D)], lambda: [r.choice(_A),"спит","дома"],
        lambda: [r.choice(_V),"едет","по","дороге"], lambda: [r.choice(_V),"стоит","у","остановки"]]
    return [r.choice(fr)() for _ in range(n)]
CATS = {w: c for c, ws in enumerate([_A, _F + _D, _V]) for w in ws}

def make_reviews(n=400, seed=0):
    r = random.Random(seed); N = ["фильм","товар","сервис","ужин","ремонт"]
    P = ["отличный","прекрасный","хороший","замечательный"]; B = ["ужасный","плохой","скучный","отвратительный"]; rows = []
    for _ in range(n):
        pos = r.random() < .5; adj = r.choice(P if pos else B); neg = r.random() < .3
        pat = r.choice(["{n} {a}","{n} очень {a}"]) if not neg else "{n} не {a}"
        rows.append((pat.format(n=r.choice(N), a=adj), int(pos != neg)))
    return pd.DataFrame(rows, columns=["text", "label"])

_PM = ["Иван","Ахмад","Фарход","Дмитрий","Алишер"]; _PF = ["Мария","Дильноза","Анна","Мадина"]
_SM = ["Петров","Каримов","Ахмедов","Иванов"]; _SF = [s + "а" for s in _SM]
_C = {"Ташкент":("Ташкент","Ташкенте"),"Самарканд":("Самарканд","Самарканде"),"Бухара":("Бухару","Бухаре"),"Москва":("Москву","Москве"),"Казань":("Казань","Казани")}
def make_ner(n=500, seed=0):
    """[(tokens, BIO-теги)]"""
    r = random.Random(seed); out = []
    for _ in range(n):
        f = r.random() < .5; c = r.choice(list(_C)); nm = r.choice(_PF if f else _PM); sn = r.choice(_SF if f else _SM)
        if r.random() < .5: t = [nm, sn, "приехала" if f else "приехал", "в", _C[c][0]]
        else: t = [nm, sn, "живёт", "в", _C[c][1]]
        out.append((t, ["B-PER","I-PER","O","O","B-LOC"]))
    return out

_PL_G = ["вокзала","аэропорта","центра","университета","рынка"]; _PL_A = ["вокзал","аэропорт","центр","университет","рынок"]
def make_intents(n=400, seed=0):
    r = random.Random(seed); rows = []
    T = {"book": ["хочу такси из {g} в {a}","закажите такси до {a}","нужно такси из {g}","вызовите такси в {a}","мне нужно такси","хочу поехать в {a}"],
         "cancel": ["отмените заказ","отмена заказа","хочу отменить такси"], "greet": ["привет","здравствуйте","добрый день"], "thanks": ["спасибо","благодарю вас","большое спасибо"]}
    for _ in range(n):
        k = r.choice(list(T)); rows.append((r.choice(T[k]).format(g=r.choice(_PL_G), a=r.choice(_PL_A)), k))
    return pd.DataFrame(rows, columns=["text", "intent"])

def make_parallel(n=200, seed=0):
    r = random.Random(seed); S = {"мама":"mother","папа":"father","девочка":"girl","студент":"student"}; V = {"читает":"reads","пишет":"writes","видит":"sees"}
    O = {"книгу":"book","газету":"newspaper","письмо":"letter","картину":"picture"}; out = []
    for _ in range(n):
        s, v, o = r.choice(list(S)), r.choice(list(V)), r.choice(list(O)); out.append(([s, v, o], [S[s], V[v], O[o]]))
    return out

ARTICLE = ["В Ташкенте открылась новая выставка современной живописи.","Выставка объединила работы тридцати художников из разных стран.",
 "Организаторы отметили, что выставка современной живописи привлекла много посетителей.","Художники представили картины, посвящённые городу и его жителям.",
 "Посетители особенно выделили работы молодых художников.","Выставка продлится до конца месяца, вход для студентов бесплатный.","Погода в городе в эти дни остаётся тёплой и солнечной."]
PASSAGES = ["Ташкент является столицей Узбекистана и крупнейшим городом страны с населением более двух миллионов человек.","Метро Ташкента открыто в 1977 году и стало первым метрополитеном в Средней Азии.",
 "Река Сырдарья протекает по территории Узбекистана, Казахстана и Таджикистана.","Самарканд известен древними мечетями и площадью Регистан.","Узбекская кухня славится пловом, который готовят в большом казане."]
QUESTIONS = [("Когда появился подземный транспорт в городе Ташкенте?", 1, "в 1977 году"), ("Какое блюдо считается визитной карточкой узбекской кухни?", 4, "плов"), ("Какая водная артерия пересекает несколько стран региона?", 2, "Сырдарья")]
AMBIG = [("Он взял ключ.", ["ключ от замка","родник","гаечный ключ"]), ("Я увидел человека с телескопом.", ["я смотрел в телескоп","у человека был телескоп"]), ("Эти стали прочнее.", ["сорта стали","сделались"])]

In [ ]:
from nlp_common import *

## 1. Дерево составляющих: КС-грамматика и chart-парсер (nltk)

In [ ]:
import nltk
from collections import Counter
sents = make_sentences(400, seed=5)
lex = {}
for s in sents:
    for w, t in zip(s["tokens"], s["tags"]): lex.setdefault(t, set()).add(w)
if LANG == "ru":
    rules = "S -> NP VP\nNP -> ADJ NOUN | NOUN\nVP -> VERB NP | VERB NP PP\nPP -> ADP NOUN"
else:
    rules = "S -> NP NP VERB | NP NP NP VERB\nNP -> ADJ NOUN | NOUN"
rules += "\n" + "\n".join(t + " -> " + " | ".join(repr(w) for w in sorted(ws)) for t, ws in lex.items())
parser = nltk.ChartParser(nltk.CFG.fromstring(rules))
parsed = sum(any(True for _ in parser.parse(s["tokens"])) for s in sents)
print('Предложений разобрано:', parsed, "/", len(sents))
print(Counter(len(list(parser.parse(s["tokens"]))) for s in sents[:200]))
tree = next(parser.parse(sents[0]["tokens"])); print(" ".join(sents[0]["tokens"])); tree.pretty_print()

## 2. Дерево зависимостей и переходный парсер (arc-standard)

In [ ]:
def heads(tags):
    n = len(tags); v = tags.index("VERB"); h = [0] * n
    for i, t in enumerate(tags):
        if t == "VERB": h[i] = 0
        elif t == "ADJ": h[i] = next(j for j in range(i + 1, n) if tags[j] == "NOUN") + 1
        elif t == "ADP": h[i] = i + 2                      # предлог зависит от своего существительного
        else: h[i] = v + 1
    return h
def oracle(h):
    n = len(h); head = [0] + h; stack, buf, arcs, acts = [0], list(range(1, n + 1)), set(), []
    done = lambda d: all(head[i] != d or i in arcs for i in range(1, n + 1))
    while buf or len(stack) > 1:
        s0 = stack[-1]; s1 = stack[-2] if len(stack) > 1 else None
        if s1 not in (None, 0) and head[s1] == s0: acts.append(("LA", s1, s0)); arcs.add(s1); stack.pop(-2)
        elif s1 is not None and head[s0] == s1 and done(s0): acts.append(("RA", s1, s0)); arcs.add(s0); stack.pop()
        else: acts.append(("SH", None, None)); stack.append(buf.pop(0))
    return acts
s = sents[3]; h = heads(s["tags"])
print(list(zip(s["tokens"], s["tags"], h)))
acts = oracle(h); print('Действия:', [a[0] for a in acts], "| len =", len(acts), "= 2n =", 2 * len(h))
rec = {a[2]: a[1] for a in acts if a[0] == "RA"}; rec.update({a[1]: a[2] for a in acts if a[0] == "LA"})
print('Дуги восстановлены верно:', all(rec.get(i + 1, 0) == h[i] or h[i] == 0 for i in range(len(h))))

## Задания

1. Добавьте в грамматику правило, допускающее предложения без прямого дополнения. Как меняется покрытие?
2. Найдите предложение с несколькими разборами (если есть) и выведите все деревья.
3. Проверьте, что оракул даёт 2n действий для любого предложения корпуса.